# Analyse av manual review-tags i mappings-JSON-filer

Les alle JSON-filer fra `mappings*`-mapper og lag plots for review-analyse.

**Sett `ROOT` til mappen der `mappings`-undermappene dine ligger.**

In [ ]:
from pathlib import Path

# ── KONFIGURASJON — endre disse ────────────────────────────────────
ROOT    = Path(".")          # rot-mappe der mappings*-mapper ligger
OUT_DIR = Path("plot_exports")  # mappe for lagring av plots og CSV
TOP_N   = 20                 # antall selskaper i top-selskaper-plottet
# ────────────────────────────────────────────────────────────────────

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Root: {ROOT.resolve()}")
print(f"Output: {OUT_DIR.resolve()}")

## Importer og plot-tema

In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

THEME = {
    "figure.facecolor": "#f5f5f7",
    "axes.facecolor": "#ebedf2",
    "axes.edgecolor": "#c9ced8",
    "axes.labelcolor": "#2f3b4a",
    "axes.titlecolor": "#1f2937",
    "axes.grid": True,
    "grid.color": "#b8c0cc",
    "grid.linestyle": "--",
    "grid.alpha": 0.45,
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 17,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.color": "#435165",
    "ytick.color": "#435165",
    "legend.frameon": False,
    "figure.autolayout": False,
}

PALETTE = {
    "blue":   "#4C78A8",
    "teal":   "#54A6A6",
    "green":  "#72B7B2",
    "orange": "#F2A65A",
    "red":    "#D95F5F",
    "gold":   "#E9C46A",
    "navy":   "#2A4E6E",
    "slate":  "#7B8BA3",
}

TAG_COLORS = {
    "CR:c":  PALETTE["red"],
    "CR:nc": PALETTE["orange"],
    "R:c":   PALETTE["blue"],
    "R:nc":  PALETTE["teal"],
    "none":  PALETTE["slate"],
}

plt.rcParams.update(THEME)
if HAS_SEABORN:
    sns.set_theme(style="darkgrid", rc=THEME)

print("Tema lastet.")

## Hjelpefunksjoner

In [ ]:
def fmt_int(x, pos=None):
    try:
        return f"{int(x):,}"
    except Exception:
        return str(x)

def style_axis(ax, title, xlabel=None, ylabel=None):
    ax.set_title(title, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_int))

def add_bar_labels(ax, fmt="{:,.0f}", pad_frac=0.01):
    heights = [p.get_height() for p in ax.patches]
    ymax = max(heights, default=0)
    offset = ymax * pad_frac if ymax > 0 else 0.1
    for patch in ax.patches:
        v = patch.get_height()
        if v > 0:
            ax.text(
                patch.get_x() + patch.get_width() / 2,
                v + offset,
                fmt.format(v),
                ha="center", va="bottom",
                fontsize=10, color="#253243", fontweight="semibold",
            )

def extract_tag(notes: str) -> str:
    if not notes:
        return "none"
    m = re.search(r'\b(CR:nc|CR:c|R:nc|R:c)\b', notes, re.IGNORECASE)
    if m:
        return m.group(1).upper().replace("NC", "nc").replace(":C", ":c")
    return "none"

def extract_cr_severity(notes: str):
    if not notes:
        return None
    m = re.search(r'CR:c\|(\w+)', notes, re.IGNORECASE)
    return m.group(1).lower() if m else None

def extract_cr_reason(notes: str):
    if not notes:
        return None
    m = re.search(r'CR:c\|\w+\|(\w+)', notes, re.IGNORECASE)
    return m.group(1).lower() if m else None

print("Hjelpefunksjoner definert.")

## Last inn data fra mappings-mapper

In [ ]:
def load_all_mappings(root: Path) -> pd.DataFrame:
    records = []
    mapping_dirs = [d for d in root.iterdir() if d.is_dir() and d.name.startswith("mappings")]
    if not mapping_dirs:
        for subdir in root.iterdir():
            if subdir.is_dir():
                mapping_dirs += [d for d in subdir.iterdir() if d.is_dir() and d.name.startswith("mappings")]
    if not mapping_dirs:
        raise FileNotFoundError(f"Fant ingen mappings*-mapper under {root}.")

    print(f"Fant {len(mapping_dirs)} mappings-mapper:")
    for folder in sorted(mapping_dirs):
        json_files = list(folder.rglob("*.json"))
        print(f"  {folder.name}: {len(json_files)} JSON-filer")
        for jf in json_files:
            try:
                data = json.loads(jf.read_text(encoding="utf-8"))
            except Exception as e:
                print(f"    ADVARSEL: {jf.name}: {e}")
                continue
            company = jf.stem
            for var_obj in data.get("variables", []):
                variable = var_obj.get("variable", "UNKNOWN")
                notes    = var_obj.get("notes", "") or ""
                needs_mr = bool(var_obj.get("needs_manual_review", False))
                candidates = var_obj.get("candidates", [])
                max_confidence = max((c.get("confidence", 0) for c in candidates), default=None)
                tag      = extract_tag(notes)
                severity = extract_cr_severity(notes)
                reason   = extract_cr_reason(notes)
                records.append({
                    "folder": folder.name, "company": company, "variable": variable,
                    "notes": notes, "tag": tag, "needs_mr": needs_mr,
                    "n_candidates": len(candidates), "n_final": len(var_obj.get("final_choice", [])),
                    "max_confidence": max_confidence, "cr_severity": severity, "cr_reason": reason,
                    "has_review": tag != "none",
                    "is_confirmed": tag in ("CR:c", "R:c"),
                    "is_cost_ratio": tag.startswith("CR"),
                })
    df = pd.DataFrame(records)
    print(f"\nTotalt {len(df)} variabel-oppføringer fra {df['company'].nunique()} selskaper.")
    return df

df = load_all_mappings(ROOT)

## Sammendrag

In [ ]:
print('\n' + '═'*55)
print('  SAMMENDRAG')
print('═'*55)
print(f"  Totale variabel-oppføringer : {len(df):,}")
print(f"  Unike selskaper             : {df['company'].nunique():,}")
print(f"  Unike mapper                : {df['folder'].nunique():,}")
print()
tag_counts = df['tag'].value_counts()
for tag in ['CR:c', 'CR:nc', 'R:c', 'R:nc', 'none']:
    n = tag_counts.get(tag, 0)
    pct = n / len(df) * 100
    print(f"  {tag:<8} : {n:>5,}  ({pct:5.1f}%)")
print()
print(f"  needs_manual_review=True    : {df['needs_mr'].sum():,}")
print('═'*55)

## Plot 1 — Manual review-tags per mappe (stacked bar)

In [ ]:
tag_order = ['CR:c', 'CR:nc', 'R:c', 'R:nc', 'none']
pivot = (
    df.groupby(['folder', 'tag'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=tag_order, fill_value=0)
)
fig, ax = plt.subplots(figsize=(12, 6))
bottom = np.zeros(len(pivot))
for tag in tag_order:
    vals = pivot[tag].values
    ax.bar(pivot.index, vals, bottom=bottom, color=TAG_COLORS[tag], label=tag, edgecolor='white', linewidth=0.8)
    bottom += vals
ax.legend(title='Tag', bbox_to_anchor=(1.01, 1), loc='upper left')
style_axis(ax, 'Manual review-tags per mappe', xlabel='Mappe', ylabel='Antall variabel-oppføringer')
ax.set_xticklabels(pivot.index, rotation=20, ha='right')
fig.tight_layout()
fig.savefig(OUT_DIR / '1_tag_overview_per_folder.png', dpi=150)
plt.show()

## Plot 2 — Review-aktivitet per variabel og mappe (heatmap)

In [ ]:
reviewed = df[df['has_review']].copy()
pivot2 = reviewed.groupby(['variable', 'folder']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(max(8, len(pivot2.columns)*1.5), max(5, len(pivot2)*0.6)))
if HAS_SEABORN:
    sns.heatmap(pivot2, annot=True, fmt='d', linewidths=0.5, cmap='Blues', ax=ax, cbar_kws={'label': 'Antall reviews'})
else:
    im = ax.imshow(pivot2.values, aspect='auto', cmap='Blues')
    ax.set_xticks(range(len(pivot2.columns)))
    ax.set_xticklabels(pivot2.columns, rotation=25, ha='right')
    ax.set_yticks(range(len(pivot2.index)))
    ax.set_yticklabels(pivot2.index)
    for i in range(len(pivot2.index)):
        for j in range(len(pivot2.columns)):
            ax.text(j, i, str(pivot2.values[i, j]), ha='center', va='center', fontsize=9)
    fig.colorbar(im, ax=ax, label='Antall reviews')
ax.set_title('Review-aktivitet per variabel og mappe', pad=12, fontsize=17, fontweight='bold')
ax.set_xlabel('Mappe')
ax.set_ylabel('Variabel')
fig.tight_layout()
fig.savefig(OUT_DIR / '2_review_heatmap_var_folder.png', dpi=150)
plt.show()

## Plot 3 — Fordeling av review-tags

- **CR:c / CR:nc** → unike selskaper (ett selskap telles én gang uansett antall flaggede variabler)
- **R:c / R:nc** → antall tilfeller (hvert enkelt sted review er gjort)

In [ ]:
tag_order3 = ['CR:c', 'CR:nc', 'R:c', 'R:nc']
counts3 = {}
for tag in ['CR:c', 'CR:nc']:
    counts3[tag] = df[df['tag'] == tag]['company'].nunique()
for tag in ['R:c', 'R:nc']:
    counts3[tag] = int((df['tag'] == tag).sum())
values3 = [counts3[t] for t in tag_order3]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(tag_order3, values3, color=[TAG_COLORS[t] for t in tag_order3], edgecolor='white', linewidth=1.2)
add_bar_labels(ax)
style_axis(ax, 'Fordeling av review-tags (alle mapper)', ylabel='Antall')
ax.axvline(1.5, color='#9ca3af', linewidth=1.2, linestyle=':')

# Bruk transAxes (relative koordinater 0-1) for å unngå at get_ylim() gir feil verdi
ax.text(0.25, 0.97, 'unike selskaper', ha='center', va='top',
        fontsize=9, color='#6b7280', style='italic', transform=ax.transAxes)
ax.text(0.75, 0.97, 'antall tilfeller', ha='center', va='top',
        fontsize=9, color='#6b7280', style='italic', transform=ax.transAxes)

legend_info = {
    'CR:c':  'Cost Ratio review — confirmed  (unike selskaper)',
    'CR:nc': 'Cost Ratio review — not confirmed  (unike selskaper)',
    'R:c':   'Review — confirmed  (tilfeller)',
    'R:nc':  'Review — not confirmed  (tilfeller)',
}
patches = [mpatches.Patch(color=TAG_COLORS[t], label=f"{t}: {legend_info[t]}") for t in tag_order3]
ax.legend(handles=patches, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR / '3_tag_distribution.png', dpi=150)
plt.show()

## Plot 4 — Andel selskaper med minst én review-tag per mappe

In [ ]:
reviewed_per_folder = df[df['has_review']].groupby('folder')['company'].nunique()
total_per_folder    = df.groupby('folder')['company'].nunique()
pct = (reviewed_per_folder / total_per_folder * 100).fillna(0).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(pct.index, pct.values, color=PALETTE['navy'], edgecolor='white', linewidth=1.2)
for patch, val in zip(ax.patches, pct.values):
    ax.text(patch.get_x() + patch.get_width()/2, val+0.5, f'{val:.1f}%',
            ha='center', va='bottom', fontsize=10, color='#253243', fontweight='semibold')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.0f}%'))
style_axis(ax, 'Andel selskaper med minst én review-tag', xlabel='Mappe', ylabel='% selskaper')
ax.set_xticklabels(pct.index, rotation=20, ha='right')
ax.set_ylim(0, min(pct.max()*1.2, 110))
fig.tight_layout()
fig.savefig(OUT_DIR / '4_review_rate_per_folder.png', dpi=150)
plt.show()

## Plot 5 — CR:c severity-fordeling

In [ ]:
cr_c = df[df['tag'] == 'CR:c'].copy()
if cr_c.empty:
    print('Ingen CR:c-tags — hopper over.')
else:
    severity_counts = cr_c['cr_severity'].fillna('(ikke spesifisert)').value_counts()
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = [PALETTE['red'] if s != '(ikke spesifisert)' else PALETTE['slate'] for s in severity_counts.index]
    ax.bar(severity_counts.index, severity_counts.values, color=colors, edgecolor='white', linewidth=1.2)
    add_bar_labels(ax)
    style_axis(ax, 'CR:c — Severity-fordeling', xlabel='Severity', ylabel='Antall')
    fig.tight_layout()
    fig.savefig(OUT_DIR / '5_cr_severity.png', dpi=150)
    plt.show()

## Plot 6 — Max kandidat-confidence etter review-tag (boxplot)

In [ ]:
sub = df[df['max_confidence'].notna() & (df['tag'] != 'none')].copy()
if sub.empty:
    print('Ikke nok data — hopper over.')
else:
    tag_order6 = [t for t in ['CR:c', 'CR:nc', 'R:c', 'R:nc'] if t in sub['tag'].unique()]
    fig, ax = plt.subplots(figsize=(9, 5))
    if HAS_SEABORN:
        sns.boxplot(data=sub, x='tag', y='max_confidence', order=tag_order6,
                    palette={t: TAG_COLORS[t] for t in tag_order6}, ax=ax, linewidth=1.5, fliersize=4)
    else:
        data_groups = [sub[sub['tag'] == t]['max_confidence'].values for t in tag_order6]
        bp = ax.boxplot(data_groups, labels=tag_order6, patch_artist=True)
        for patch, tag in zip(bp['boxes'], tag_order6):
            patch.set_facecolor(TAG_COLORS[tag])
    style_axis(ax, 'Max kandidat-confidence etter review-tag', xlabel='Tag', ylabel='Max confidence (0–1)')
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.2f}'))
    fig.tight_layout()
    fig.savefig(OUT_DIR / '6_confidence_by_tag.png', dpi=150)
    plt.show()

## Plot 7 — Top-N selskaper etter antall review-tags

In [ ]:
reviewed7 = df[df['has_review']].copy()
if reviewed7.empty:
    print('Ingen review-tags — hopper over.')
else:
    counts7 = reviewed7.groupby(['company', 'tag']).size().unstack(fill_value=0)
    tag_order7 = [t for t in ['CR:c', 'CR:nc', 'R:c', 'R:nc'] if t in counts7.columns]
    counts7['total'] = counts7[tag_order7].sum(axis=1)
    top = counts7.nlargest(TOP_N, 'total')[tag_order7]
    fig, ax = plt.subplots(figsize=(13, 6))
    bottom = np.zeros(len(top))
    for tag in tag_order7:
        ax.bar(top.index, top[tag].values, bottom=bottom, color=TAG_COLORS[tag],
               label=tag, edgecolor='white', linewidth=0.7)
        bottom += top[tag].values
    ax.legend(title='Tag', bbox_to_anchor=(1.01, 1), loc='upper left')
    style_axis(ax, f'Top-{TOP_N} selskaper etter antall review-tags', xlabel='Selskap', ylabel='Antall review-tags')
    ax.set_xticklabels(top.index, rotation=35, ha='right', fontsize=9)
    fig.tight_layout()
    fig.savefig(OUT_DIR / '7_top_companies_by_reviews.png', dpi=150)
    plt.show()

## Lagre flat CSV for videre analyse

In [ ]:
csv_path = OUT_DIR / 'mappings_review_flat.csv'
df.to_csv(csv_path, index=False)
print(f'CSV lagret: {csv_path}')
print(f'Kolonner: {list(df.columns)}')
df.head()